Concurrency, The GIL & Shared Memory
Core Mechanics & Theory
To parallelize RL rollout environments, stream data to GPU buffers, and build multi-core pipelines, you must understand how CPython handles execution at the OS thread and process levels.

First: what is the GIL?

GIL = Global Interpreter Lock.

In standard CPython, the GIL means:

At a given moment, only one thread can execute Python bytecode within a Python interpreter.

Suppose you have:

def compute():
    total = 0
    for i in range(10_000_000):
        total += i

This is CPU-bound Python work.

If you create:

Thread 1 → compute()
Thread 2 → compute()

you might expect:

CPU Core 1 → Thread 1
CPU Core 2 → Thread 2

But for Python bytecode, the GIL prevents both threads from executing Python bytecode simultaneously.

Conceptually:

Thread 1: ████████     ████████
Thread 2:         ████████     ████████
                 ↑
              GIL moves

So you don't get true parallel execution of the Python bytecode.

2. Why does Python have the GIL?

This is where the technical side matters.

CPython uses reference counting for memory management.

An object has a reference count conceptually like:

object
  │
  └── refcount = 3

When another variable references it:

refcount += 1

When a reference disappears:

refcount -= 1

These operations need to be safely coordinated between threads.

The GIL historically provides a simple mechanism to protect CPython's interpreter state and memory-management machinery.

So:

GIL isn't fundamentally a "threading feature"; it's a CPython implementation mechanism that restricts simultaneous execution of Python bytecode.

Also, one important modern nuance: free-threaded CPython builds exist, so "Python always has a GIL" is no longer universally true. But for the standard CPython setup you're likely using, the GIL model in your exercise is the right one to learn.

Threading vs multiprocessing

This is the most important distinction.

Thread
threading.Thread(...)

Threads live inside the same process.

So:

Process
│
├── Thread 1
├── Thread 2
└── Thread 3

They share the process's memory.

That's convenient:

data = [1, 2, 3]

Both threads can access data.

But shared memory means you can get:

Race conditions

and may need synchronization primitives such as:

Lock
Process
multiprocessing.Process(...)

creates another OS process.

Conceptually:

Process 1
    Memory A


Process 2
    Memory B

They don't normally share Python objects directly.

That's why processes can execute Python bytecode truly in parallel on multiple CPU cores.

4. When should you use each?

The simple rule:

CPU-bound Python
      ↓
multiprocessing


I/O-bound work
      ↓
threading / asyncio

For example:

CPU-bound
huge_python_calculation()

Processes are usually the better choice.

I/O-bound
download_file()
wait_for_network()
read_from_socket()

Threads can be useful because while one thread is waiting for I/O, another can run.

GIL benchmark

Let's build exactly what your exercise asks.

In [1]:
import time
import threading
import multiprocessing


def compute_heavy(n: int):
    return sum(i * i for i in range(n))

In [3]:
N = 10_000_000

start = time.perf_counter()

compute_heavy(N)
compute_heavy(N)

elapsed = time.perf_counter() - start

print(f"Sequential: {elapsed:.2f}s")

Sequential: 1.35s


In [4]:
def run_thread():
    compute_heavy(N)
t1 = threading.Thread(target=run_thread)
t2 = threading.Thread(target=run_thread)

start = time.perf_counter()

t1.start()
t2.start()

t1.join()
t2.join()

elapsed = time.perf_counter() - start

print(f"Threads: {elapsed:.2f}s")

Threads: 1.54s


What do we expect?

You might expect:

Sequential:
Task A ██████████
Task B           ██████████


Threads:
Task A ██████████
Task B ██████████

and therefore threads should be twice as fast.

But for CPU-bound Python:

Threads:
A ███  ███  ███  ███
B   ███  ███  ███  ███

The threads take turns executing Python bytecode because of the GIL.

Therefore you might see something roughly like:

Sequential: 1.5 sec
Threads:    1.6 sec

or even worse depending on your machine.

In [6]:
def run_process():
    compute_heavy(N)

In [7]:
p1 = multiprocessing.Process(target=run_process)
p2 = multiprocessing.Process(target=run_process)

start = time.perf_counter()

p1.start()
p2.start()

p1.join()
p2.join()

elapsed = time.perf_counter() - start

print(f"Processes: {elapsed:.2f}s")

Processes: 0.16s
